In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
import torch
from transformers import BertTokenizer, BertModel
import numpy as np
from tqdm import tqdm



c:\Users\kotam\anaconda3\envs\tf-env\lib\site-packages\transformers\utils\generic.py:260: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [2]:
# Load the dataset
data = pd.read_csv("..\data\CyberTrollIEEE.csv")

In [3]:
# Preprocessing function (simple lowercasing, remove special characters)
def preprocess_text(text):
    text = text.lower()  # Lowercase text
    return text

In [4]:
# Preprocess the text data
data['cleaned_content'] = data['content'].apply(preprocess_text)

In [5]:
# Split the dataset into features (X) and target (y)
X = data['cleaned_content']
y = data['annotation']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [6]:
# Load BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

c:\Users\kotam\anaconda3\envs\tf-env\lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [12]:
import numpy as np
import torch
from tqdm import tqdm

# Function to convert text into BERT embeddings
def get_bert_embeddings(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)  # Mean of all token embeddings
    return embeddings.numpy().flatten()

# Generate and save embeddings only if not already saved
try:
    print("Loading saved embeddings...")
    train_embeddings = np.load("train_embeddings.npy")
    test_embeddings = np.load("test_embeddings.npy")
    print("Embeddings loaded successfully!")
except FileNotFoundError:
    print("Generating embeddings (this may take some time)...")
    train_embeddings = np.array([get_bert_embeddings(text) for text in tqdm(X_train)])
    test_embeddings = np.array([get_bert_embeddings(text) for text in tqdm(X_test)])

    # Save the generated embeddings
    np.save("train_embeddings.npy", train_embeddings)
    np.save("test_embeddings.npy", test_embeddings)
    print("Embeddings saved successfully!")


Loading saved embeddings...
Generating embeddings (this may take some time)...


100%|██████████| 4429/4429 [02:17<00:00, 32.27it/s]


Embeddings saved successfully!


In [13]:
# Initialize the Random Forest model
rf_model = RandomForestClassifier(random_state=42)

In [14]:

# Define parameter grid for fine-tuning (using GridSearchCV for hyperparameter tuning)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2']
}

In [17]:
import xgboost as xgb

# Define the XGBoost model using GPU
xgb_model = xgb.XGBClassifier(tree_method="hist", device="cuda")

# Train the model
xgb_model.fit(train_embeddings, y_train)

# Predict
y_pred = xgb_model.predict(test_embeddings)


In [18]:
from sklearn.metrics import accuracy_score, classification_report

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Print classification report (Precision, Recall, F1-score)
print(classification_report(y_test, y_pred))


Accuracy: 0.9124
              precision    recall  f1-score   support

           0       0.95      0.88      0.92      2445
           1       0.87      0.95      0.91      1984

    accuracy                           0.91      4429
   macro avg       0.91      0.92      0.91      4429
weighted avg       0.92      0.91      0.91      4429



In [19]:
import joblib

# Save the model
joblib.dump(xgb_model, "../models/xgb_cyberbullying_model.pkl")
print("Model saved successfully!")


Model saved successfully!


In [28]:
import pickle
from sklearn.naive_bayes import MultinomialNB

# Train your MultinomialNB model
model = MultinomialNB()  # Example, replace with your trained model

with open('../MultinomialNB.pkl', 'rb') as file:
    modelNB = pickle.load(file)



c:\Users\kotam\anaconda3\envs\tf-env\lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 0.23.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [32]:
import numpy as np
import joblib  # or use pickle if you're using pickle for saving/loading models

# Load the trained model (replace with your actual model filename)
modelNB = joblib.load('../custom_ml_model.pkl')  # Change this if using a different model or file

# Function to test the model manually
def predict_bullying_rf(text):
    # Ensure the input is valid
    if not isinstance(text, str) or text.strip() == "":
        return "Invalid input: Please enter a valid text."

    # Preprocess the input text (cleaning, tokenization, etc.)
    cleaned_text = preprocess_text(text)

    # Get BERT embeddings for the input text
    embedding = get_bert_embeddings(cleaned_text)

    # Ensure embeddings are correctly reshaped before prediction
    embedding = np.array(embedding).reshape(1, -1)  # reshape to (1, n_features)

    # Use the trained Random Forest model to make a prediction
    prediction = modelNB.predict(embedding)

    # Return the result based on the model prediction
    if prediction == 1:
        return "Bullying"
    else:
        return "Not Bullying"

# Test with sample sentences
test_texts = [
    "You are such a loser!",  # Likely bullying
    "Hey, how are you today?",  # Not bullying
    "I hope you die",  # Likely bullying
    "That's a great idea!"  # Not bullying
]

# Run the predictions
for text in test_texts:
    result = predict_bullying_rf(text)
    print(f"Text: {text}\nPrediction: {result}\n")


ModuleNotFoundError: No module named 'sklearn.svm.classes'